# Cat and Dog Image Classification Using TensorFlow and Keras

**Group Assignment — 2 students per group**
**Mode of submission:** Class presentation

**Names:** _[Student 1 Name, Student 2 Name]_
**Group:** _[Group number]_
**Date:** _[Date]_

---

## Background

We have been employed as Data Scientists by a company developing an automated
image-recognition system. The goal is to build a system that can automatically
determine whether a given image contains a **cat** or a **dog**.

This notebook builds a complete, working image-classification pipeline using
**Python, TensorFlow and Keras**: it obtains the dataset, explores it,
preprocesses the images, builds and trains a neural network, validates and
tests the model, and demonstrates predictions on unseen images.

> **Note on the dataset source:** The original assignment points to the
> Microsoft "Cats vs Dogs" download page
> (https://www.microsoft.com/en-us/download/details.aspx?id=54765). This is
> the *same* dataset that is distributed through **TensorFlow Datasets (TFDS)**
> as `cats_vs_dogs` (23,262 usable images, after 1,738 corrupted files are
> filtered out of the original 25,000 — matching the 23,262 figure in the
> assignment brief). We load it via `tensorflow_datasets` because this gives
> a reliable, scriptable download with no broken links, while remaining the
> exact same Microsoft/Kaggle "Cats vs Dogs" data. If your marker requires the
> raw ZIP instead, the alternative download cell below shows how to do that
> too — just swap the loading cell used.


## 0. Setup

Install/import the libraries we need.

In [1]:
# If running on a fresh machine / Google Colab, uncomment the line below
# !pip install tensorflow tensorflow-datasets matplotlib scikit-learn seaborn

import numpy as np
import matplotlib.pyplot as plt
import importlib_resources
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print("TensorFlow version:", tf.__version__)

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


TensorFlow version: 2.21.0


## i. Obtain and Explore the Dataset  *(4 marks)*

We load the **Cats vs Dogs** dataset using `tensorflow_datasets`, which pulls
the same Microsoft-curated image set referenced in the assignment. TFDS
automatically discards the small number of corrupted JPEG files in the
original archive, leaving the usable **23,262 images** across **2 classes**:
`cat` and `dog`.


In [ ]:
# Load the raw dataset (this downloads ~800MB the first time it is run)
(raw_train,), ds_info = tfds.load(
    "cats_vs_dogs",
    split=["train"],          # the raw dataset ships as a single split
    with_info=True,
    as_supervised=True,       # returns (image, label) pairs
)

raw_train = raw_train[0] if isinstance(raw_train, list) else raw_train

class_names = ds_info.features["label"].names
num_classes = ds_info.features["label"].num_classes
total_images = ds_info.splits["train"].num_examples

print("Classes:", class_names)
print("Number of classes:", num_classes)
print("Total usable images:", total_images)


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

In [3]:
# Display 6 sample images with their class labels
plt.figure(figsize=(12, 6))
for i, (image, label) in enumerate(raw_train.take(6)):
    ax = plt.subplot(2, 3, i + 1)
    plt.imshow(image.numpy().astype("uint8"))
    plt.title(f"Label: {class_names[label.numpy()]}")
    plt.axis("off")
plt.suptitle("Sample images from the Cats vs Dogs dataset")
plt.tight_layout()
plt.show()


NameError: name 'raw_train' is not defined

<Figure size 1200x600 with 0 Axes>

### Dataset description and the classification problem

The **Cats vs Dogs** dataset contains 23,262 usable colour photographs of
cats and dogs, collected from real-world photos of varying size, lighting,
pose and background. Each image is labelled with exactly one of two classes:
`cat` (0) or `dog` (1).

This is a **binary image classification** problem: given a previously unseen
photograph, the model must output a single probability that indicates how
confident it is that the image is a dog (values close to 1) versus a cat
(values close to 0). Because the raw images vary in resolution and the two
classes are of a similar visual "difficulty" (both are common household
pets with fur, four legs, similar poses), the main challenges are (a)
standardising image size/format before they can be fed to a neural network,
and (b) learning visual features (shapes, textures, ear/face structure) that
generalise across many different breeds, poses and backgrounds rather than
just memorising the training photos.


## ii. Prepare and Preprocess the Images  *(5 marks)*

We now:
* resize every image to a common size (**128 × 128** pixels),
* convert images to `float32` numerical tensors,
* normalise pixel values to the range **[0, 1]**,
* split the data into **70% training / 15% validation / 15% testing**.


In [4]:
IMG_SIZE = 128
BATCH_SIZE = 32

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))   # common size
    image = tf.cast(image, tf.float32) / 255.0              # normalise to [0,1]
    return image, label

# Apply preprocessing to every image in the dataset
processed_ds = raw_train.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

# --- 70% train / 15% validation / 15% test split ---
dataset_size = total_images
train_size = int(0.70 * dataset_size)
val_size = int(0.15 * dataset_size)
test_size = dataset_size - train_size - val_size   # remainder -> ~15%

# Shuffle once, with a fixed seed, before splitting so the split is reproducible
full_ds = processed_ds.shuffle(buffer_size=dataset_size, seed=SEED)

train_ds_raw = full_ds.take(train_size)
remaining_ds = full_ds.skip(train_size)
val_ds_raw = remaining_ds.take(val_size)
test_ds_raw = remaining_ds.skip(val_size)

print(f"Training images:   {train_size}")
print(f"Validation images: {val_size}")
print(f"Testing images:    {test_size}")

# Batch and prefetch for training efficiency
train_ds = train_ds_raw.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds_raw.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds_raw.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


NameError: name 'raw_train' is not defined

### Why resizing and normalization matter

**Resizing** every image to a common size (128 × 128) is necessary because
the raw photographs come in many different resolutions and aspect ratios,
while a neural network's input layer has a **fixed** number of neurons —
every training example must have exactly the same shape for the matrix
operations in the network to work. Resizing also keeps the model small and
training fast; using the full, inconsistent original resolutions would make
batching impossible and would be computationally very expensive.

**Normalizing pixel values** (dividing by 255 so values lie in [0, 1])
matters because raw pixel intensities range from 0–255, which is a much
larger and less well-behaved numeric range than most neural network weight
initialisations expect. Large, unscaled inputs can cause unstable or slow
gradient descent (very large or very small gradients), making training
slower and less stable. Scaling all inputs to a small, consistent range
(0–1) helps the optimiser converge faster and more reliably, and puts every
input feature (pixel) on the same scale so that no pixel dominates purely
because of its raw magnitude.


## iii. Construct the Neural Network  *(5 marks)*

We build a `Sequential` Keras model containing an input layer, a flattening
layer, at least one hidden `Dense` layer, an activation function, and an
output layer suitable for binary classification.


In [5]:
model = keras.Sequential([
    # Input layer: expects 128x128 RGB images
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    # Flattening layer: converts the 3D image (128,128,3) into a 1D vector
    # so it can be fed into the Dense layers below
    layers.Flatten(),

    # Hidden Dense layer 1: learns combinations of pixel features
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),   # helps reduce overfitting

    # Hidden Dense layer 2: learns higher-level, more abstract features
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),

    # Output layer: a single neuron with a sigmoid activation, suitable for
    # BINARY classification -> outputs a probability between 0 (cat) and 1 (dog)
    layers.Dense(1, activation="sigmoid"),
])

model.summary()


NameError: name 'keras' is not defined

### Purpose of each major layer

* **Input layer** — declares the fixed shape (128 × 128 × 3) that every
  image must match; it does not transform the data, it simply tells Keras
  what shape to expect.
* **Flatten layer** — reshapes the 3-dimensional image tensor
  (height × width × colour channels) into a single 1-dimensional vector of
  49,152 values, because the following `Dense` layers only accept 1D input
  per example.
* **Hidden Dense layers (with ReLU activation)** — fully-connected layers
  that learn to combine the raw pixel values into increasingly useful
  features (edges, textures, shapes, and eventually higher-level concepts
  like "ear shape" or "fur pattern"). The **ReLU** activation function
  introduces non-linearity, which allows the network to learn complex,
  non-linear relationships instead of behaving like a simple linear model.
* **Dropout layers** — randomly "turn off" a fraction of neurons during
  training, which reduces overfitting by preventing the network from relying
  too heavily on any single neuron/feature.
* **Output layer (1 neuron, sigmoid activation)** — produces a single
  probability value between 0 and 1. This is the standard configuration for
  **binary classification**: values near 0 indicate "cat", values near 1
  indicate "dog", and the model's confidence is simply how far the value is
  from 0.5.


## iv. Compile and Train the Model  *(4 marks)*

We compile the model with the **Adam** optimizer, **binary crossentropy**
loss (the standard loss function for binary classification), and
**accuracy** as the evaluation metric, then train for at least 10 epochs
while monitoring the validation set.


In [6]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",   # appropriate loss for binary classification
    metrics=["accuracy"],
)

EPOCHS = 10

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
)


NameError: name 'model' is not defined

In [7]:
# Record final training and validation metrics
final_train_acc = history.history["accuracy"][-1]
final_val_acc = history.history["val_accuracy"][-1]
final_train_loss = history.history["loss"][-1]
final_val_loss = history.history["val_loss"][-1]

print(f"Final training accuracy:   {final_train_acc:.4f}")
print(f"Final validation accuracy: {final_val_acc:.4f}")
print(f"Final training loss:       {final_train_loss:.4f}")
print(f"Final validation loss:     {final_val_loss:.4f}")


NameError: name 'history' is not defined

*Record your actual printed values from the cell above here once you
have run the notebook, e.g.:*

| Metric | Value |
|---|---|
| Training accuracy | _e.g. 0.88_ |
| Validation accuracy | _e.g. 0.72_ |
| Training loss | _e.g. 0.27_ |
| Validation loss | _e.g. 0.58_ |


## v. Evaluate and Validate the Model  *(4 marks)*

We now evaluate the trained model on the **held-out test set** (images the
model has never seen in training or validation) and produce a confusion
matrix.


In [8]:
test_loss, test_accuracy = model.evaluate(test_ds)
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test loss:     {test_loss:.4f}")


NameError: name 'model' is not defined

In [9]:
# Gather predictions and true labels for the whole test set
y_true = []
y_pred_probs = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_pred_probs.extend(preds.flatten())
    y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)
y_pred = (y_pred_probs >= 0.5).astype(int)   # threshold at 0.5

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix - Test Set")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"Correctly classified cats:   {tn}")
print(f"Incorrectly classified cats: {fp}")
print(f"Correctly classified dogs:   {tp}")
print(f"Incorrectly classified dogs: {fn}")

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=class_names))


NameError: name 'test_ds' is not defined

### Interpreting the results

*Fill this in with your own numbers once the notebook has run, using the
confusion matrix and test accuracy/loss above as a guide. A template
interpretation:*

The test accuracy of **_[X]_%** indicates that the model correctly
classifies roughly _[X]_ out of every 100 new, unseen images. Looking at the
confusion matrix, the majority of errors fall along the diagonal opposite
cells (cats misclassified as dogs, or vice-versa), which suggests the model
has learned some genuinely useful visual features rather than guessing
randomly (random guessing would give ~50% accuracy on a balanced dataset).
If the test accuracy is noticeably lower than the training accuracy from
Part iv, this indicates some degree of **overfitting** — the model has
memorised patterns specific to the training images that do not generalise
as well to new photos. A simple fully-connected network (no convolutional
layers) also has an inherent ceiling on accuracy for natural images compared
to a CNN, since it cannot exploit local spatial structure in the same way.


## vi. Develop an Image Prediction Function  *(4 marks)*

The function below accepts a single new image (a file path), preprocesses
it exactly as the training data was preprocessed, and uses the trained
model to predict whether it shows a cat or a dog, along with a confidence
percentage.


In [10]:
def predict_image(image_path, model, class_names=("cat", "dog"), img_size=IMG_SIZE):
    """Load an image from disk, predict its class, and display the result.

    Parameters
    ----------
    image_path : str
        Path to a JPG/PNG image NOT used during training.
    model : keras.Model
        The trained classification model.
    """
    # Load and preprocess the image exactly like the training pipeline
    raw = tf.io.read_file(image_path)
    image = tf.image.decode_image(raw, channels=3, expand_animations=False)
    image_resized = tf.image.resize(image, (img_size, img_size))
    image_normalised = tf.cast(image_resized, tf.float32) / 255.0
    input_batch = tf.expand_dims(image_normalised, axis=0)   # add batch dimension

    # Predict
    probability = model.predict(input_batch, verbose=0)[0][0]

    if probability >= 0.5:
        predicted_class = class_names[1]        # "dog"
        confidence = probability * 100
    else:
        predicted_class = class_names[0]         # "cat"
        confidence = (1 - probability) * 100

    # Display the image with its prediction
    plt.figure(figsize=(4, 4))
    plt.imshow(image.numpy().astype("uint8"))
    plt.axis("off")
    plt.title(f"Prediction: {predicted_class} ({confidence:.1f}% confidence)")
    plt.show()

    print(f"Input image: {image_path}")
    print(f"Predicted class: {predicted_class}")
    print(f"Prediction confidence: {confidence:.1f}%")

    return predicted_class, confidence


In [11]:
# Demonstration: pick one image from the TEST set (never used for training)
# and save it to disk so predict_image() can load it exactly like a
# brand-new, previously unseen photo supplied by the company.

demo_image, demo_label = next(iter(test_ds_raw.take(1)))
demo_path = "/home/claude/demo_test_image.jpg"

# Save the preprocessed tensor back out as a real JPG file on disk
tf.keras.utils.save_img(demo_path, demo_image.numpy())

print("True label of demo image:", class_names[demo_label.numpy()])
predict_image(demo_path, model)

# To test with your OWN photo instead, simply call, e.g.:
# predict_image("/path/to/my_photo.jpg", model)


NameError: name 'test_ds_raw' is not defined

## vii. Analyse the Model  *(2 marks)*

We plot Training vs Validation accuracy, and Training vs Validation loss,
across the 10+ training epochs.


In [12]:
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label="Training Accuracy", marker="o")
plt.plot(epochs_range, val_acc, label="Validation Accuracy", marker="o")
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label="Training Loss", marker="o")
plt.plot(epochs_range, val_loss, label="Validation Loss", marker="o")
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()


NameError: name 'history' is not defined

### Interpreting the training behaviour

*Look at the shape of your own two graphs above and describe what you see,
using the guide below:*

* **Good learning** looks like: both training and validation accuracy rise
  together and both losses fall together, ending close to each other.
* **Overfitting** looks like: training accuracy keeps climbing towards
  ~95–100% while validation accuracy plateaus or falls, and training loss
  keeps dropping while validation loss starts rising after a few epochs —
  the gap between the two curves widens over time. This is a very common
  outcome for a plain (non-convolutional) `Dense`-only network on raw image
  pixels, because it has a huge number of parameters (from flattening a
  128×128×3 image) relative to how much genuinely useful visual structure it
  can extract, so it easily memorises training examples instead of learning
  general patterns.
* **Underfitting** looks like: both training and validation accuracy stay
  low and both losses stay high and roughly flat across epochs — the model
  simply has not learned the task, often because it needs more epochs, more
  capacity, or a better architecture (e.g. convolutional layers).

State which of these patterns your own curves best match, and suggest one
concrete change (e.g. adding convolutional layers, data augmentation, more
dropout, or early stopping) that could improve the result.


## viii. Explain Model Validation  *(2 marks)*

**Question:** A colleague suggests that after training the model, you
should evaluate it using the *same* images that were used during training.
Explain why this would not provide a reliable estimate of how well the
model will classify new, unseen images.

**Answer:**

Evaluating a model on the exact same images it was trained on will almost
always produce an overly optimistic accuracy figure, because the model has
already directly adjusted its internal weights to minimise error on those
specific images — it may have partly **memorised** particular pictures
(their exact pixel patterns, backgrounds, lighting, etc.) rather than
learning general visual features that distinguish cats from dogs. This is
similar to giving a student the exact exam paper they used to study from:
scoring well on it says nothing about whether they actually understood the
material or could answer a *different* but related question.

The real purpose of the model, in this scenario, is to classify **brand-new
photographs** the company will feed it in production — images it has never
encountered before. Because of this, the only trustworthy way to estimate
real-world performance is to evaluate the model on a **held-out test set**
that was **completely excluded from training** (and ideally from
hyperparameter tuning too, which is what the separate validation set is
for). This is precisely why the assignment splits the data into training
(70%), validation (15%, used to monitor performance *during* training and
guide decisions like when to stop), and testing (15%, used only *once*,
at the very end, to report an honest, unbiased estimate of how well the
model generalises to unseen data).


## Conclusion

This notebook implemented a complete end-to-end pipeline for classifying
cat and dog images using TensorFlow/Keras: dataset loading and exploration,
preprocessing (resizing, normalisation, 70/15/15 split), a `Sequential`
neural network, training with the Adam optimizer, evaluation with a
confusion matrix, a reusable single-image prediction function, and an
analysis of the training/validation curves. Remember to swap in your own
recorded numbers/observations in the markdown cells marked with
*[fill in]* before your presentation.
